In [2]:
"""
predict.py
----------
Prediction module for the trained 8-category news classifier.
Use this in app.py, notebooks, or standalone.

Usage:
    from predict import NewsClassifier
    
    clf = NewsClassifier()
    result = clf.predict(title="RBI raises repo rate", text="The Reserve Bank...")
    print(result)
"""

import torch
import numpy as np
import json
import re
import os
from dataclasses import dataclass
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── Default paths — change to match your setup ─────────────────────────────────
DEFAULT_MODEL_PATH     = "improved_model/best_model"
DEFAULT_TOKENIZER_PATH = "improved_model/best_tokenizer"
DEFAULT_THRESHOLD_PATH = "improved_model/best_thresholds.json"

CATEGORIES = [
    'Politics', 'Science and Technology', 'Business and Economy',
    'International Affairs', 'Education', 'Health', 'Sports', 'Entertainment'
]

# Default thresholds — replaced by Optuna values if available
DEFAULT_THRESHOLDS = [
    0.6251,   # Politics
    0.7554,   # Science and Technology
    0.4783,   # Business and Economy
    0.7679,   # International Affairs
    0.5961,   # Education
    0.7345,   # Health
    0.6283,   # Sports
    0.7477,   # Entertainment
]


@dataclass
class PredictionResult:
    """Structured result from the classifier."""
    labels:        list[str]          # predicted category names
    probabilities: dict[str, float]   # calibrated probability per label
    confidence:    float              # mean probability of predicted labels
    is_uncertain:  bool               # True if max prob < uncertainty_threshold
    raw_scores:    dict[str, float]   # raw sigmoid scores before thresholding


class NewsClassifier:
    """
    Multi-label news article classifier.
    Loads once, call predict() as many times as needed.
    """

    def __init__(
        self,
        model_path:     str = DEFAULT_MODEL_PATH,
        tokenizer_path: str = DEFAULT_TOKENIZER_PATH,
        threshold_path: str = DEFAULT_THRESHOLD_PATH,
        device:         str = 'auto',
        max_length:     int = 256,
        temperature:    float = 1.0,    # calibration temperature (1.0 = no calibration)
    ):
        # Device
        if device == 'auto':
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = torch.device(device)

        self.max_length  = max_length
        self.temperature = temperature

        # Load tokenizer and model
        print(f"Loading model from: {model_path}")
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        self.model     = AutoModelForSequenceClassification.from_pretrained(model_path)
        self.model.to(self.device)
        self.model.eval()
        print(f"Model loaded on {self.device}")

        # Load Optuna thresholds if available, else use defaults
        self.thresholds = self._load_thresholds(threshold_path)

        # Category → index map
        self.cat_to_idx = {cat: i for i, cat in enumerate(CATEGORIES)}

    def _load_thresholds(self, path: str) -> list[float]:
        """Load per-label thresholds from Optuna output, or use defaults."""
        if os.path.exists(path):
            with open(path) as f:
                data = json.load(f)
            thresholds = data.get('threshold_list', DEFAULT_THRESHOLDS)
            print(f"Loaded Optuna thresholds from: {path}")
            return thresholds
        else:
            print(f"No threshold file found at {path}. Using default 0.5.")
            return DEFAULT_THRESHOLDS

    def _clean_text(self, text: str) -> str:
        """Minimal cleaning — strip HTML, normalize whitespace."""
        if not isinstance(text, str):
            return ""
        text = re.sub(r'<[^>]+>', ' ', text)
        text = re.sub(r'https?://\S+', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    @torch.no_grad()
    def predict(
        self,
        title:                str,
        text:                 str,
        uncertainty_threshold: float = 0.15,
        return_all_scores:    bool  = False,
    ) -> PredictionResult:
        """
        Classify a news article.

        Args:
            title:                  Article headline
            text:                   Article body text
            uncertainty_threshold:  If max probability < this, mark as uncertain
                                    and fall back to zero-shot OOD detection
            return_all_scores:      If True, include all 8 scores in result

        Returns:
            PredictionResult with labels, probabilities, confidence flag
        """
        # Preprocess
        clean_title = self._clean_text(title)
        clean_text  = self._clean_text(text)

        # Combine title + body (same format as training)
        combined = f"{clean_title} [SEP] {clean_text}"

        # Tokenize
        encoding = self.tokenizer(
            combined,
            max_length        = self.max_length,
            padding           = 'max_length',
            truncation        = True,
            return_tensors    = 'pt',
            return_attention_mask = True,
        )

        input_ids      = encoding['input_ids'].to(self.device)
        attention_mask = encoding['attention_mask'].to(self.device)

        # Forward pass
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits.squeeze(0)  # shape: (8,)

        # Apply temperature calibration
        if self.temperature != 1.0:
            logits = logits / self.temperature

        # Sigmoid → probabilities
        probs = torch.sigmoid(logits).cpu().numpy()

        # Apply per-label thresholds
        predicted_indices = [
            i for i, (p, t) in enumerate(zip(probs, self.thresholds))
            if p > t
        ]

        # Always predict at least one label (highest probability)
        if not predicted_indices:
            predicted_indices = [int(np.argmax(probs))]

        predicted_labels = [CATEGORIES[i] for i in predicted_indices]

        # Build probability dict (all 8 labels)
        all_probs = {cat: round(float(probs[i]), 4)
                     for i, cat in enumerate(CATEGORIES)}

        # Confidence = mean prob of predicted labels
        confidence = float(np.mean([probs[i] for i in predicted_indices]))

        # Uncertainty flag
        is_uncertain = float(np.max(probs)) < uncertainty_threshold

        return PredictionResult(
            labels        = predicted_labels,
            probabilities = {cat: all_probs[cat] for cat in predicted_labels},
            confidence    = round(confidence, 4),
            is_uncertain  = is_uncertain,
            raw_scores    = all_probs if return_all_scores else {},
        )

    def predict_batch(
        self,
        articles: list[dict],
        batch_size: int = 32,
    ) -> list[PredictionResult]:
        """
        Classify multiple articles efficiently.

        Args:
            articles:   List of dicts with 'title' and 'text' keys
            batch_size: Number of articles to process at once

        Returns:
            List of PredictionResult, one per article
        """
        results = []

        for batch_start in range(0, len(articles), batch_size):
            batch = articles[batch_start : batch_start + batch_size]

            texts = [
                f"{self._clean_text(a.get('title',''))} [SEP] "
                f"{self._clean_text(a.get('text',''))}"
                for a in batch
            ]

            encodings = self.tokenizer(
                texts,
                max_length        = self.max_length,
                padding           = 'max_length',
                truncation        = True,
                return_tensors    = 'pt',
                return_attention_mask = True,
            )

            input_ids      = encodings['input_ids'].to(self.device)
            attention_mask = encodings['attention_mask'].to(self.device)

            with torch.no_grad():
                logits = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                ).logits

                if self.temperature != 1.0:
                    logits = logits / self.temperature

                probs_batch = torch.sigmoid(logits).cpu().numpy()

            for probs in probs_batch:
                predicted_indices = [
                    i for i, (p, t) in enumerate(zip(probs, self.thresholds))
                    if p > t
                ] or [int(np.argmax(probs))]

                predicted_labels = [CATEGORIES[i] for i in predicted_indices]
                all_probs        = {cat: round(float(probs[i]), 4)
                                    for i, cat in enumerate(CATEGORIES)}

                results.append(PredictionResult(
                    labels        = predicted_labels,
                    probabilities = {cat: all_probs[cat] for cat in predicted_labels},
                    confidence    = round(float(np.mean([probs[i] for i in predicted_indices])), 4),
                    is_uncertain  = float(np.max(probs)) < 0.15,
                    raw_scores    = all_probs,
                ))

        return results

    def predict_from_url(self, url: str) -> PredictionResult:
        """
        Scrape a URL and classify it.
        Requires: pip install newspaper4k lxml_html_clean
        """
        try:
            from newspaper import Article, Config

            config = Config()
            config.fetch_images = False
            config.request_timeout = 12

            article = Article(url, config=config)
            article.download()
            article.parse()

            if len(article.text.split()) < 50:
                raise ValueError("Article too short — may be paywalled.")

            return self.predict(
                title = article.title or '',
                text  = article.text,
            )

        except ImportError:
            raise ImportError("Install newspaper4k: pip install newspaper4k lxml_html_clean")


# ── Standalone usage ────────────────────────────────────────────────────────────

def predict_single(title: str, text: str,
                   model_path: str = DEFAULT_MODEL_PATH,
                   tokenizer_path: str = DEFAULT_TOKENIZER_PATH) -> dict:
    """
    Simple one-shot function for quick testing.
    Creates a new classifier instance each call — use NewsClassifier()
    directly if making multiple predictions.
    """
    clf    = NewsClassifier(model_path, tokenizer_path)
    result = clf.predict(title, text, return_all_scores=True)
    return {
        'labels':      result.labels,
        'confidence':  result.confidence,
        'all_scores':  result.raw_scores,
        'is_uncertain': result.is_uncertain,
    }


# ── Test / demo ────────────────────────────────────────────────────────────────

if __name__ == '__main__':
    # Load classifier once
    clf = NewsClassifier()

    # Test articles
    test_articles = [
        {
            'title': 'RBI raises repo rate by 25 basis points to control inflation',
            'text':  'The Reserve Bank of India on Friday raised its benchmark '
                     'lending rate by 25 basis points to 6.75%, the sixth '
                     'consecutive hike, as it battles persistently high inflation '
                     'driven by food prices and global commodity costs. The monetary '
                     'policy committee voted 5-1 in favour of the rate hike.',
        },
        {
            'title': 'India vs Australia: Virat Kohli scores century in 3rd Test',
            'text':  'Virat Kohli smashed his 30th Test century as India dominated '
                     'Australia on Day 2 of the third Test in Melbourne. Kohli '
                     'remained unbeaten on 115 as India reached 342/5 at stumps, '
                     'with KL Rahul providing solid support with 78 runs.',
        },
        {
            'title': 'New study links ultra-processed food consumption to dementia risk',
            'text':  'Researchers at University College London have found a significant '
                     'association between high consumption of ultra-processed foods and '
                     'increased risk of dementia. The study, published in Neurology, '
                     'followed 72,000 participants over a decade and found a 25% '
                     'higher risk among those in the top quartile of UPF consumption.',
        },
        {
            'title': 'OpenAI announces GPT-5 with enhanced reasoning capabilities',
            'text':  'OpenAI has released GPT-5, claiming significant improvements in '
                     'mathematical reasoning, coding, and multimodal understanding. '
                     'The model scores 92% on the MMLU benchmark, up from 87% for '
                     'GPT-4. Enterprise pricing starts at $60 per million tokens.',
        },
        {
            'title': 'Parliament passes new education policy amid teacher protests',
            'text':  'The Lok Sabha on Thursday passed the National Education Policy '
                     'Amendment Bill 2025 after hours of heated debate. The bill '
                     'proposes mandatory vocational training from Class 9 and '
                     'introduces a new 5+3+3+4 curricular structure. Teachers unions '
                     'across five states called for a strike, citing inadequate '
                     'implementation funds.',
        },
    ]

    print(f"\n{'='*65}")
    print(f"CLASSIFICATION RESULTS")
    print(f"{'='*65}")

    for article in test_articles:
        result = clf.predict(
            title             = article['title'],
            text              = article['text'],
            return_all_scores = True,
        )

        print(f"\nTitle: {article['title'][:65]}")
        print(f"Labels:     {result.labels}")
        print(f"Confidence: {result.confidence:.1%}")

        if result.is_uncertain:
            print(f"WARNING: Low confidence — article may be outside training domain")

        print(f"All scores:")
        # Sort by score descending for readability
        sorted_scores = sorted(
            result.raw_scores.items(), key=lambda x: x[1], reverse=True
        )
        for cat, score in sorted_scores:
            bar    = '█' * int(score * 20)
            marker = ' ← predicted' if cat in result.labels else ''
            print(f"  {cat:<28} {score:.3f}  {bar}{marker}")

        print()

Loading model from: improved_model/best_model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded on cpu
Loaded Optuna thresholds from: improved_model/best_thresholds.json

CLASSIFICATION RESULTS

Title: RBI raises repo rate by 25 basis points to control inflation
Labels:     ['Business and Economy']
Confidence: 95.8%
All scores:
  Business and Economy         0.958  ███████████████████ ← predicted
  Health                       0.410  ████████
  Entertainment                0.378  ███████
  Education                    0.369  ███████
  International Affairs        0.358  ███████
  Sports                       0.336  ██████
  Science and Technology       0.240  ████
  Politics                     0.197  ███


Title: India vs Australia: Virat Kohli scores century in 3rd Test
Labels:     ['Sports']
Confidence: 98.6%
All scores:
  Sports                       0.986  ███████████████████ ← predicted
  Entertainment                0.405  ████████
  Health                       0.372  ███████
  Education                    0.330  ██████
  International Affairs        0.320  █